# Nigeria Incidents, Accidents & Violence — Data Cleaning
**Dataset:** [Kaggle – Incidents, Accidents and Violence in Nigeria](https://www.kaggle.com/datasets/jogwums/incidents-accidents-and-violence-in-nigeria)  
**Goal:** Clean and prepare the dataset before Exploratory Data Analysis (EDA)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
print('Libraries loaded successfully ✅')

## 2. Load the Dataset

In [ ]:
# Update the path below to wherever you saved the downloaded CSV
df = pd.read_csv('nigeria_incidents.csv', encoding='utf-8', low_memory=False)

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

## 3. Initial Inspection

In [ ]:
# Column names and data types
print('=== Column Info ===')
df.info()

In [ ]:
# Basic statistics
print('=== Descriptive Statistics ===')
df.describe(include='all')

In [ ]:
# Check column names for whitespace or inconsistency
print('Column names:')
print(df.columns.tolist())

## 4. Standardise Column Names

In [ ]:
# Strip whitespace, lowercase, replace spaces with underscores
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace(r'[^\w]', '_', regex=True)
)

print('Cleaned column names:')
print(df.columns.tolist())

## 5. Check & Handle Missing Values

In [ ]:
# Count and percentage of missing values per column
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['missing_count'] > 0].sort_values('missing_pct', ascending=False)
print(missing)

# Visual
plt.figure(figsize=(10, 5))
sns.barplot(x=missing.index, y=missing['missing_pct'], palette='Reds_r')
plt.xticks(rotation=45, ha='right')
plt.title('Missing Values by Column (%)')
plt.ylabel('Missing %')
plt.tight_layout()
plt.show()

In [ ]:
# Drop columns where more than 60% of data is missing
threshold = 0.6
cols_to_drop = missing[missing['missing_pct'] > threshold * 100].index.tolist()
print(f'Columns to drop (>{threshold*100}% missing): {cols_to_drop}')

df.drop(columns=cols_to_drop, inplace=True)
print(f'Shape after dropping high-missing columns: {df.shape}')

In [ ]:
# For remaining columns — fill or flag missing values
# Categorical columns: fill with 'Unknown'
cat_cols = df.select_dtypes(include='object').columns
df[cat_cols] = df[cat_cols].fillna('Unknown')

# Numeric columns: fill with median (robust to outliers)
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
for col in num_cols:
    median_val = df[col].median()
    df[col].fillna(median_val, inplace=True)

print('Missing values after treatment:')
print(df.isnull().sum().sum())

## 6. Check & Remove Duplicate Rows

In [ ]:
dupes = df.duplicated().sum()
print(f'Duplicate rows found: {dupes}')

df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape after removing duplicates: {df.shape}')

## 7. Fix Data Types

In [ ]:
# --- DATE COLUMNS ---
# Identify columns that look like dates
date_candidates = [col for col in df.columns if any(kw in col for kw in ['date', 'year', 'month', 'time'])]
print(f'Potential date columns: {date_candidates}')

for col in date_candidates:
    try:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f'  Converted "{col}" to datetime')
    except Exception as e:
        print(f'  Could not convert "{col}": {e}')

In [ ]:
# Extract useful date parts if a datetime column exists
datetime_cols = df.select_dtypes(include='datetime64').columns.tolist()

for col in datetime_cols:
    df[f'{col}_year']  = df[col].dt.year
    df[f'{col}_month'] = df[col].dt.month
    df[f'{col}_month_name'] = df[col].dt.month_name()
    df[f'{col}_day_of_week'] = df[col].dt.day_name()
    print(f'Extracted year, month, month_name, day_of_week from "{col}"')

In [ ]:
# --- NUMERIC COLUMNS ---
# Force numeric conversion on columns that should be numbers
suspected_numeric = [col for col in df.columns if any(kw in col for kw in ['count', 'killed', 'injured', 'fatalities', 'casualties', 'number', 'total', 'deaths'])]
print(f'Suspected numeric columns: {suspected_numeric}')

for col in suspected_numeric:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

## 8. Standardise Text / Categorical Columns

In [ ]:
# Strip leading/trailing whitespace, title-case all text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().str.title()

print('Text columns standardised ✅')

In [ ]:
# Check unique values in key categorical columns
# Adjust column names below to match your actual dataset
key_cats = [col for col in df.columns if any(kw in col for kw in ['state', 'type', 'region', 'zone', 'incident', 'category'])]

for col in key_cats:
    print(f'\n--- {col} ---')
    print(df[col].value_counts().head(15))

## 9. Handle Outliers in Numeric Columns

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Visualise distributions
if len(num_cols) > 0:
    df[num_cols].hist(figsize=(14, 8), bins=30, edgecolor='black', color='steelblue')
    plt.suptitle('Distribution of Numeric Columns', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cap outliers using IQR method (Winsorizing)
def cap_outliers(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    return series.clip(lower=lower, upper=upper)

# Apply only to non-year/month numeric columns
skip_keywords = ['year', 'month', 'day', 'id']
outlier_cols = [c for c in num_cols if not any(kw in c for kw in skip_keywords)]

for col in outlier_cols:
    before = df[col].max()
    df[col] = cap_outliers(df[col])
    after = df[col].max()
    if before != after:
        print(f'Capped "{col}": max changed from {before} → {after}')

print('\nOutlier capping complete ✅')

## 10. Validate Nigerian States

In [ ]:
# Official list of 36 Nigerian states + FCT
NIGERIAN_STATES = {
    'Abia', 'Adamawa', 'Akwa Ibom', 'Anambra', 'Bauchi', 'Bayelsa',
    'Benue', 'Borno', 'Cross River', 'Delta', 'Ebonyi', 'Edo', 'Ekiti',
    'Enugu', 'Fct', 'Gombe', 'Imo', 'Jigawa', 'Kaduna', 'Kano', 'Katsina',
    'Kebbi', 'Kogi', 'Kwara', 'Lagos', 'Nasarawa', 'Niger', 'Ogun',
    'Ondo', 'Osun', 'Oyo', 'Plateau', 'Rivers', 'Sokoto', 'Taraba', 'Yobe', 'Zamfara'
}

# Detect the state column
state_col = next((col for col in df.columns if 'state' in col), None)

if state_col:
    invalid_states = df[~df[state_col].isin(NIGERIAN_STATES)][state_col].unique()
    print(f'Invalid / unrecognised state entries: {invalid_states}')
else:
    print('No state column detected. Update the column name manually if needed.')

## 11. Final Cleaned Dataset — Summary

In [ ]:
print('=== CLEANED DATASET SUMMARY ===')
print(f'Rows     : {df.shape[0]:,}')
print(f'Columns  : {df.shape[1]}')
print(f'Missing  : {df.isnull().sum().sum()}')
print(f'Duplicates: {df.duplicated().sum()}')
print()
print(df.dtypes)

In [ ]:
df.head(10)

## 12. Export Cleaned Dataset

In [ ]:
df.to_csv('nigeria_incidents_cleaned.csv', index=False)
print('Cleaned dataset saved as nigeria_incidents_cleaned.csv ✅')
print('You can now load this file for EDA and visualisation.')